In [1]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import pandas as pd
import numpy as np

RA_ROOT = Path(r"C:/Users/Carl/Desktop/RA Data")
CAND_ROOT = RA_ROOT / "11K CANDIDATES"
PROJECT = Path(r"C:/Users/Carl/Desktop/CSB_project")

PATHS = {
    "villages": CAND_ROOT / "Villages outside 30 km CSB buffers.xlsx",
    "merge_indices": CAND_ROOT / "merge indices.xlsx",
    "hh_panel": PROJECT / "Built panels" / "hh_panel_roster_9_10_11_12_13_17_18.xlsx",
    "agsec10": PROJECT / "Finished sections" / "Agriculture" / "AGSEC10_wide.csv",
    "ag3": PROJECT / "Built panels" / "AG3_inputs_aggregated.xlsx",
}

BUFFER_KMS = [10, 20, 30, 40]
WAVE_KEEP = 1


def load_table(path, sheet_name=0):
    path = Path(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name, dtype=str)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=str)
    raise ValueError(f"Unsupported file type: {path.suffix}")


def clean_id(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return pd.NA
    try:
        if "e" in s.lower():
            s = format(Decimal(s), "f")
        if s.endswith(".0"):
            s = s[:-2]
    except InvalidOperation:
        pass
    return s


def clean_wave(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower().replace("wave", "").replace("_", "").replace("-", "")
    return int(float(s))


def add_standard_keys(df, wave_col=None, hhid_col=None, hh_id_obs_col=None):
    df = df.copy()
    if wave_col and wave_col in df.columns:
        df["Wave_clean"] = df[wave_col].map(clean_wave).astype("Int64")
    if hhid_col and hhid_col in df.columns:
        df["HHID_clean"] = df[hhid_col].map(clean_id)
    if hh_id_obs_col and hh_id_obs_col in df.columns:
        df["hh_id_obs_clean"] = df[hh_id_obs_col].map(clean_id)
    return df


def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"WARNING: {name} missing columns:", missing)
    return [c for c in cols if c in df.columns]

In [2]:
# Load eligible villages and attach geonameid

villages = load_table(PATHS["villages"])
villages.columns = villages.columns.str.strip()

villages = villages.rename(columns={"Index": "index"})
villages["index_clean"] = villages["index"].map(clean_id)

merge_indices = load_table(PATHS["merge_indices"])
merge_indices.columns = merge_indices.columns.str.strip()
merge_indices = merge_indices.rename(columns={"Index": "index"})
merge_indices["index_clean"] = merge_indices["index"].map(clean_id)
merge_indices["geonameid_clean"] = merge_indices["geonameid"].map(clean_id)

merge_indices_map = (
    merge_indices[["index_clean", "geonameid", "geonameid_clean"]]
    .drop_duplicates("index_clean")
)

villages = villages.merge(
    merge_indices_map,
    on="index_clean",
    how="left",
    validate="m:1",
)

print("Eligible villages:", villages.shape)
print("Missing geonameid:", villages["geonameid_clean"].isna().sum())
display(villages.head())


# Load HH panel, wave 1 only

hh_panel = load_table(PATHS["hh_panel"])
hh_panel = add_standard_keys(
    hh_panel,
    wave_col="Wave",
    hhid_col="HHID",
    hh_id_obs_col="hh_id_obs",
)

hh_panel = hh_panel[hh_panel["Wave_clean"].eq(WAVE_KEEP)].copy()

hh_vars_wanted = [
    "Wave", "HHID", "hh_id_obs",
    "GSEC15A__TOTAL_HH_MEMBERS_15A",
    "GSEC12__H12Q01",
    "H11Q01",
    "GSEC10__H10Q1",
    "GSEC10__H10Q09",
    "H18Q1A", "H18Q1B", "H18Q1C", "H18Q1D",
    "H18Q4A", "H18Q4B", "H18Q4C", "H18Q4D",
    "GSEC17__H17Q9",
    "GSEC17__H17Q10",
    "GSEC17__H17Q11",
    "Wave_clean", "HHID_clean", "hh_id_obs_clean",
]

hh_panel_small = hh_panel[require_columns(hh_panel, hh_vars_wanted, "HH panel")].copy()

print("Duplicate hh_id_obs in wave 1 HH panel:",
      hh_panel_small.duplicated("hh_id_obs_clean").sum())

hh_panel_small = hh_panel_small.drop_duplicates("hh_id_obs_clean")


# Load AGSEC10, wave 1 only

agsec10 = load_table(PATHS["agsec10"])
agsec10 = add_standard_keys(agsec10, wave_col="Wave", hhid_col="HHID")
agsec10 = agsec10[agsec10["Wave_clean"].eq(WAVE_KEEP)].copy()

agsec10_vars = [
    "Wave", "HHID",
    "AGSEC10_ANY", "AGSEC10_PROD", "AGSEC10_PRICES", "AGSEC10_PROC",
    "Wave_clean", "HHID_clean",
]

agsec10_small = (
    agsec10[require_columns(agsec10, agsec10_vars, "AGSEC10")]
    .drop_duplicates(["Wave_clean", "HHID_clean"])
)


# Load AG3 and pivot visits wide, wave 1 only

ag3 = load_table(PATHS["ag3"])
ag3 = add_standard_keys(ag3, wave_col="Wave", hhid_col="HHID")
ag3 = ag3[ag3["Wave_clean"].eq(WAVE_KEEP)].copy()
ag3["VISIT"] = pd.to_numeric(ag3["VISIT"], errors="coerce").astype("Int64")

ag3_value_cols = [
    "A3Q4_average_use",
    "A3Q14_average_use",
    "A3Q26_average_use",
    "A3AQ38_average",
    "A3Q39_average",
    "A3Q41_any_labor",
    "A3Q43_sum",
]

ag3_value_cols = require_columns(ag3, ag3_value_cols, "AG3")

for col in ag3_value_cols:
    ag3[col] = pd.to_numeric(ag3[col], errors="coerce")

ag3_collapsed = (
    ag3
    .groupby(["Wave_clean", "HHID_clean", "VISIT"], as_index=False)
    .agg({col: "mean" if col != "A3Q43_sum" else "sum" for col in ag3_value_cols})
)

ag3_name_map = {
    "A3Q4_average_use": "A3Q4",
    "A3Q14_average_use": "A3Q14",
    "A3Q26_average_use": "A3Q26",
    "A3AQ38_average": "A3AQ38",
    "A3Q39_average": "A3Q39",
    "A3Q41_any_labor": "A3Q41",
    "A3Q43_sum": "A3Q43",
}

ag3_wide_parts = []

for source_col, prefix in ag3_name_map.items():
    part = (
        ag3_collapsed
        .pivot(index=["Wave_clean", "HHID_clean"], columns="VISIT", values=source_col)
        .reindex(columns=[1, 2])
    )
    part.columns = [f"{prefix}_{int(v)}" for v in part.columns]
    ag3_wide_parts.append(part)

ag3_wide = pd.concat(ag3_wide_parts, axis=1).reset_index()

print("Setup ready.")

Eligible villages: (9147, 7)
Missing geonameid: 0


,index,name_11k,latitude,longitude,index_clean,geonameid,geonameid_clean
0,2,Zombo,2.51355,30.90909,2,225797,225797
1,3,Ziru,0.14106,32.52202,3,225803,225803
2,4,Zirobwe,0.67944,32.68944,4,225805,225805
3,5,Zira,0.44509,32.1608,5,225809,225809
4,6,Zintengeze,0.5,32.98333,6,225810,225810


Duplicate hh_id_obs in wave 1 HH panel: 0
Setup ready.


In [3]:
def clean_csb_name(x):
    if pd.isna(x):
        return pd.NA
    return " ".join(str(x).strip().upper().split())


def prepare_household_vars(src):
    src = src.copy()

    key_cols = {
        "site_id", "site_type", "site_name",
        "index", "name_11k", "geonameid", "geonameid_clean",
        "buffer_km", "treated",
        "HHID", "HHID_clean",
        "hh_id_obs", "hh_id_obs_clean",
        "hh_id_obs_hhpanel",
        "Wave", "Wave_clean",
    }

    for col in src.columns:
        if col not in key_cols:
            converted = pd.to_numeric(src[col], errors="coerce")
            if converted.notna().any():
                src[col] = converted

    if "AGSEC10_ANY" in src.columns:
        matched_hh = src["HHID_clean"].notna()
        src["AGSEC10_ANY"] = pd.to_numeric(src["AGSEC10_ANY"], errors="coerce")
        src.loc[matched_hh, "AGSEC10_ANY"] = src.loc[matched_hh, "AGSEC10_ANY"].fillna(0)

    if "H11Q01" in src.columns:
        src["avg_subsistence"] = np.where(
            src["H11Q01"].notna(),
            src["H11Q01"].eq(4).astype(float),
            np.nan,
        )
        src["non_agriculture_enterprise_avg"] = pd.to_numeric(src["H11Q01"], errors="coerce")

    if "GSEC10__H10Q1" in src.columns:
        src["electricity_access_avg"] = pd.to_numeric(src["GSEC10__H10Q1"], errors="coerce")

    if "A3AQ38_1" in src.columns:
        src.loc[src["A3AQ38_1"].eq(8336.3), "A3AQ38_1"] = pd.NA

    hh_size_col = "GSEC15A__TOTAL_HH_MEMBERS_15A"

    if hh_size_col in src.columns:
        hh_size = pd.to_numeric(src[hh_size_col], errors="coerce")

        for visit in [1, 2]:
            src_col = f"A3AQ38_{visit}"
            out_col = f"share_hh_working_{visit}"

            if src_col in src.columns:
                src[out_col] = pd.to_numeric(src[src_col], errors="coerce") / hh_size
                src.loc[hh_size.le(0) | hh_size.isna(), out_col] = pd.NA

    src["hh_id_obs_lsms_matched"] = src["hh_id_obs_clean"].where(src["HHID_clean"].notna())

    return src


def mean_sd_aggregate(src):
    src = src.copy()

    group_keys = [
        "site_id", "site_type", "treated", "site_name",
        "index", "name_11k", "geonameid",
        "buffer_km",
    ]

    exclude_from_stats = set(group_keys) | {
        "Wave", "Wave_clean",
        "HHID", "HHID_clean",
        "hh_id_obs", "hh_id_obs_clean",
        "hh_id_obs_hhpanel",
        "hh_id_obs_lsms_matched",
        "geonameid_clean",
        "H11Q01",
        "GSEC10__H10Q1",
    }

    numeric_for_stats = [
        c for c in src.select_dtypes(include="number").columns
        if c not in exclude_from_stats
    ]

    agg_spec = {}

    for col in numeric_for_stats:
        agg_spec[f"{col}_mean"] = (col, "mean")
        agg_spec[f"{col}_sd"] = (col, "std")

    agg_spec["n_hh_id_obs_buffer"] = ("hh_id_obs_clean", pd.Series.nunique)
    agg_spec["n_hh_id_obs_lsms_matched"] = ("hh_id_obs_lsms_matched", pd.Series.nunique)

    out = (
        src
        .groupby(group_keys, as_index=False, dropna=False)
        .agg(**agg_spec)
    )

    out["Wave"] = WAVE_KEEP

    front_cols = [
        "site_id", "site_type", "treated", "site_name",
        "index", "name_11k", "geonameid",
        "buffer_km", "Wave",
        "n_hh_id_obs_buffer",
        "n_hh_id_obs_lsms_matched",
    ]

    front_cols = [c for c in front_cols if c in out.columns]
    other_cols = [c for c in out.columns if c not in front_cols]

    return out[front_cols + other_cols]

In [4]:
def build_combined_buffer_panel(km):
    buffer_path = RA_ROOT / f"geonameid_csb_hh_wave1_panel_{km}KM.csv"

    buffer_hh = load_table(buffer_path)
    buffer_hh.columns = buffer_hh.columns.str.strip()

    buffer_hh["site_raw"] = buffer_hh["geonameid_CSB"].astype("string").str.strip()
    buffer_hh["hh_id_obs_clean"] = buffer_hh["hh_id_obs"].map(clean_id)

    is_numeric_site = buffer_hh["site_raw"].str.fullmatch(r"\d+(\.0)?", na=False)

    village_hh = buffer_hh[is_numeric_site].copy()
    csb_hh = buffer_hh[~is_numeric_site].copy()

    # -------------------------
    # Control villages
    # -------------------------
    village_hh["geonameid_clean"] = village_hh["site_raw"].map(clean_id)

    village_base = villages.merge(
        village_hh[["geonameid_clean", "hh_id_obs", "hh_id_obs_clean"]],
        on="geonameid_clean",
        how="left",
    )

    village_base["buffer_km"] = km
    village_base["treated"] = 0
    village_base["site_type"] = "village"
    village_base["site_id"] = "VILLAGE_" + village_base["index"].astype("string")
    village_base["site_name"] = village_base["name_11k"]

    village_panel = village_base.merge(
        hh_panel_small,
        on="hh_id_obs_clean",
        how="left",
        validate="m:1",
        suffixes=("", "_hhpanel"),
    )

    village_panel = village_panel.merge(
        agsec10_small.drop(columns=["Wave", "HHID"], errors="ignore"),
        on=["Wave_clean", "HHID_clean"],
        how="left",
        validate="m:1",
    )

    village_panel = village_panel.merge(
        ag3_wide,
        on=["Wave_clean", "HHID_clean"],
        how="left",
        validate="m:1",
    )

    village_panel = prepare_household_vars(village_panel)
    village_out = mean_sd_aggregate(village_panel)

    # -------------------------
    # CSBs
    # -------------------------
    csb_hh["CSB_clean"] = csb_hh["site_raw"].map(clean_csb_name)
    csb_hh = csb_hh.drop_duplicates(["CSB_clean", "hh_id_obs_clean"])

    csb_base = csb_hh[["site_raw", "CSB_clean", "hh_id_obs", "hh_id_obs_clean"]].copy()

    csb_base["buffer_km"] = km
    csb_base["treated"] = 1
    csb_base["site_type"] = "csb"
    csb_base["site_id"] = "CSB_" + csb_base["CSB_clean"]
    csb_base["site_name"] = csb_base["site_raw"]

    csb_base["index"] = pd.NA
    csb_base["name_11k"] = pd.NA
    csb_base["geonameid"] = pd.NA
    csb_base["geonameid_clean"] = pd.NA

    csb_panel = csb_base.merge(
        hh_panel_small,
        on="hh_id_obs_clean",
        how="left",
        validate="m:1",
        suffixes=("", "_hhpanel"),
    )

    csb_panel = csb_panel.merge(
        agsec10_small.drop(columns=["Wave", "HHID"], errors="ignore"),
        on=["Wave_clean", "HHID_clean"],
        how="left",
        validate="m:1",
    )

    csb_panel = csb_panel.merge(
        ag3_wide,
        on=["Wave_clean", "HHID_clean"],
        how="left",
        validate="m:1",
    )

    csb_panel = prepare_household_vars(csb_panel)
    csb_out = mean_sd_aggregate(csb_panel)

    combined = pd.concat(
        [village_out, csb_out],
        ignore_index=True,
        sort=False,
    )

    print(f"\n{km}KM combined panel")
    print("Village sites:", village_out["site_id"].nunique())
    print("CSB sites:", csb_out["site_id"].nunique())
    print("Rows:", len(combined))

    return combined

In [5]:
OUT = CAND_ROOT / "buffer_panels_wave1"
OUT.mkdir(exist_ok=True)

combined_buffer_panels = {}

for km in BUFFER_KMS:
    panel_km = build_combined_buffer_panel(km)
    combined_buffer_panels[km] = panel_km

    out_file = OUT / f"combined_villages_csbs_wave1_lsms_{km}KM.xlsx"
    panel_km.to_excel(out_file, index=False)

    print("Saved:", out_file)
    display(panel_km.head())


combined_all_buffers = pd.concat(
    combined_buffer_panels.values(),
    ignore_index=True,
    sort=False,
)

out_file_all = OUT / "combined_villages_csbs_wave1_lsms_all_buffers.xlsx"
combined_all_buffers.to_excel(out_file_all, index=False)

print("Saved stacked file:", out_file_all)


10KM combined panel
Village sites: 9147
CSB sites: 8
Rows: 9155
Saved: C:\Users\Carl\Desktop\RA Data\11K CANDIDATES\buffer_panels_wave1\combined_villages_csbs_wave1_lsms_10KM.xlsx


,site_id,site_type,treated,site_name,index,name_11k,geonameid,buffer_km,Wave,n_hh_id_obs_buffer,...,avg_subsistence_mean,avg_subsistence_sd,non_agriculture_enterprise_avg_mean,non_agriculture_enterprise_avg_sd,electricity_access_avg_mean,electricity_access_avg_sd,share_hh_working_1_mean,share_hh_working_1_sd,share_hh_working_2_mean,share_hh_working_2_sd
0,VILLAGE_10,village,0,Ziba,10,Ziba,225826,10,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,VILLAGE_10000,village,0,Masimbi,10000,Masimbi,11552102,10,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,VILLAGE_10001,village,0,Magoggo,10001,Magoggo,11552103,10,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,VILLAGE_10002,village,0,Namagera,10002,Namagera,11552104,10,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VILLAGE_10003,village,0,Busonko,10003,Busonko,11552105,10,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



20KM combined panel
Village sites: 9147
CSB sites: 16
Rows: 9163
Saved: C:\Users\Carl\Desktop\RA Data\11K CANDIDATES\buffer_panels_wave1\combined_villages_csbs_wave1_lsms_20KM.xlsx


,site_id,site_type,treated,site_name,index,name_11k,geonameid,buffer_km,Wave,n_hh_id_obs_buffer,...,avg_subsistence_mean,avg_subsistence_sd,non_agriculture_enterprise_avg_mean,non_agriculture_enterprise_avg_sd,electricity_access_avg_mean,electricity_access_avg_sd,share_hh_working_1_mean,share_hh_working_1_sd,share_hh_working_2_mean,share_hh_working_2_sd
0,VILLAGE_10,village,0,Ziba,10,Ziba,225826,20,1,1,...,0.0,NaN,5.0,NaN,0.0,NaN,0.107143,NaN,0.155844,NaN
1,VILLAGE_10000,village,0,Masimbi,10000,Masimbi,11552102,20,1,5,...,0.2,0.447214,3.2,2.48998,0.0,0.0,0.592593,0.21033,0.607407,0.055925
2,VILLAGE_10001,village,0,Magoggo,10001,Magoggo,11552103,20,1,5,...,0.2,0.447214,3.2,2.48998,0.0,0.0,0.592593,0.21033,0.607407,0.055925
3,VILLAGE_10002,village,0,Namagera,10002,Namagera,11552104,20,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,VILLAGE_10003,village,0,Busonko,10003,Busonko,11552105,20,1,5,...,0.2,0.447214,3.2,2.48998,0.0,0.0,0.592593,0.21033,0.607407,0.055925



30KM combined panel
Village sites: 9147
CSB sites: 17
Rows: 9164
Saved: C:\Users\Carl\Desktop\RA Data\11K CANDIDATES\buffer_panels_wave1\combined_villages_csbs_wave1_lsms_30KM.xlsx


,site_id,site_type,treated,site_name,index,name_11k,geonameid,buffer_km,Wave,n_hh_id_obs_buffer,...,avg_subsistence_mean,avg_subsistence_sd,non_agriculture_enterprise_avg_mean,non_agriculture_enterprise_avg_sd,electricity_access_avg_mean,electricity_access_avg_sd,share_hh_working_1_mean,share_hh_working_1_sd,share_hh_working_2_mean,share_hh_working_2_sd
0,VILLAGE_10,village,0,Ziba,10,Ziba,225826,30,1,38,...,0.405405,0.497743,3.000000,2.027588,0.088235,0.287902,0.618606,0.589791,0.545286,0.416754
1,VILLAGE_10000,village,0,Masimbi,10000,Masimbi,11552102,30,1,59,...,0.293103,0.459161,2.896552,1.860859,0.236364,0.428764,0.577220,0.661237,0.477400,0.257678
2,VILLAGE_10001,village,0,Magoggo,10001,Magoggo,11552103,30,1,58,...,0.280702,0.453336,2.877193,1.871498,0.236364,0.428764,0.577220,0.661237,0.477400,0.257678
3,VILLAGE_10002,village,0,Namagera,10002,Namagera,11552104,30,1,60,...,0.288136,0.456782,2.898305,1.844796,0.236364,0.428764,0.575188,0.652361,0.471570,0.256859
4,VILLAGE_10003,village,0,Busonko,10003,Busonko,11552105,30,1,59,...,0.293103,0.459161,2.896552,1.860859,0.236364,0.428764,0.577220,0.661237,0.477400,0.257678



40KM combined panel
Village sites: 9147
CSB sites: 19
Rows: 9166
Saved: C:\Users\Carl\Desktop\RA Data\11K CANDIDATES\buffer_panels_wave1\combined_villages_csbs_wave1_lsms_40KM.xlsx


,site_id,site_type,treated,site_name,index,name_11k,geonameid,buffer_km,Wave,n_hh_id_obs_buffer,...,avg_subsistence_mean,avg_subsistence_sd,non_agriculture_enterprise_avg_mean,non_agriculture_enterprise_avg_sd,electricity_access_avg_mean,electricity_access_avg_sd,share_hh_working_1_mean,share_hh_working_1_sd,share_hh_working_2_mean,share_hh_working_2_sd
0,VILLAGE_10,village,0,Ziba,10,Ziba,225826,40,1,72,...,0.295775,0.459639,2.661972,1.715007,0.138462,0.348072,0.652498,0.588440,0.577420,0.369754
1,VILLAGE_10000,village,0,Masimbi,10000,Masimbi,11552102,40,1,88,...,0.244186,0.432123,2.930233,1.896068,0.243590,0.432026,0.545355,0.567388,0.485656,0.276199
2,VILLAGE_10001,village,0,Magoggo,10001,Magoggo,11552103,40,1,88,...,0.244186,0.432123,2.930233,1.896068,0.243590,0.432026,0.545355,0.567388,0.485656,0.276199
3,VILLAGE_10002,village,0,Namagera,10002,Namagera,11552104,40,1,99,...,0.278351,0.450515,2.979381,1.814177,0.275862,0.449539,0.556961,0.554890,0.490335,0.272526
4,VILLAGE_10003,village,0,Busonko,10003,Busonko,11552105,40,1,89,...,0.241379,0.430400,2.931034,1.885027,0.243590,0.432026,0.545355,0.567388,0.485656,0.276199


Saved stacked file: C:\Users\Carl\Desktop\RA Data\11K CANDIDATES\buffer_panels_wave1\combined_villages_csbs_wave1_lsms_all_buffers.xlsx


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT = CAND_ROOT / "buffer_panels_wave1"

try:
    balance_source = combined_all_buffers.copy()
except NameError:
    balance_source = pd.read_excel(
        OUT / "combined_villages_csbs_wave1_lsms_all_buffers.xlsx"
    )

balance_source["treated"] = pd.to_numeric(balance_source["treated"], errors="coerce")
balance_source["buffer_km"] = pd.to_numeric(balance_source["buffer_km"], errors="coerce")
balance_source["n_hh_id_obs_buffer"] = pd.to_numeric(
    balance_source["n_hh_id_obs_buffer"],
    errors="coerce"
)

# Drop sites with zero household observations in that buffer
balance_source = balance_source[
    balance_source["n_hh_id_obs_buffer"].fillna(0).gt(0)
].copy()

# Use site-level means as balance variables
balance_vars = [
    c for c in balance_source.columns
    if c.endswith("_mean")
    and not c.startswith("n_")
]

print("Balance variables:", len(balance_vars))
print(balance_vars)

Balance variables: 38
['latitude_mean', 'longitude_mean', 'index_clean_mean', 'GSEC15A__TOTAL_HH_MEMBERS_15A_mean', 'GSEC12__H12Q01_mean', 'GSEC10__H10Q09_mean', 'H18Q1A_mean', 'H18Q1B_mean', 'H18Q1C_mean', 'H18Q1D_mean', 'H18Q4A_mean', 'H18Q4B_mean', 'H18Q4C_mean', 'H18Q4D_mean', 'GSEC17__H17Q9_mean', 'AGSEC10_ANY_mean', 'AGSEC10_PROD_mean', 'AGSEC10_PRICES_mean', 'AGSEC10_PROC_mean', 'A3Q4_1_mean', 'A3Q4_2_mean', 'A3Q14_1_mean', 'A3Q14_2_mean', 'A3Q26_1_mean', 'A3Q26_2_mean', 'A3AQ38_1_mean', 'A3AQ38_2_mean', 'A3Q39_1_mean', 'A3Q39_2_mean', 'A3Q41_1_mean', 'A3Q41_2_mean', 'A3Q43_1_mean', 'A3Q43_2_mean', 'avg_subsistence_mean', 'non_agriculture_enterprise_avg_mean', 'electricity_access_avg_mean', 'share_hh_working_1_mean', 'share_hh_working_2_mean']


In [7]:
def variable_label(col):
    base = col.removesuffix("_mean")

    labels = {
        "avg_subsistence": "Subsistence primary income",
        "non_agriculture_enterprise_avg": "Non-agricultural enterprise",
        "electricity_access_avg": "Electricity access",
        "AGSEC10_ANY": "Agricultural extension advice",
        "AGSEC10_PROD": "Extension: production",
        "AGSEC10_PRICES": "Extension: prices",
        "AGSEC10_PROC": "Extension: processing",
        "H18Q1A": "Road type A",
        "H18Q1B": "Road type B",
        "H18Q1C": "Road type C",
        "H18Q1D": "Road type D",
        "H18Q4A": "Time to road A",
        "H18Q4B": "Time to road B",
        "H18Q4C": "Time to road C",
        "H18Q4D": "Time to road D",
        "A3Q4_1": "Pesticide use, season 1",
        "A3Q4_2": "Pesticide use, season 2",
        "A3Q14_1": "Organic fertilizer, season 1",
        "A3Q14_2": "Organic fertilizer, season 2",
        "A3Q26_1": "Inorganic fertilizer, season 1",
        "A3Q26_2": "Inorganic fertilizer, season 2",
        "A3Q39_1": "Days on plots, season 1",
        "A3Q39_2": "Days on plots, season 2",
        "A3Q41_1": "Hired labor, season 1",
        "A3Q41_2": "Hired labor, season 2",
        "A3Q43_1": "Labor cash/in-kind, season 1",
        "A3Q43_2": "Labor cash/in-kind, season 2",
        "share_hh_working_1": "Share HH working, season 1",
        "share_hh_working_2": "Share HH working, season 2",
    }

    return labels.get(base, base)


def unweighted_stats(df, var):
    t = pd.to_numeric(df.loc[df["treated"].eq(1), var], errors="coerce").dropna()
    c = pd.to_numeric(df.loc[df["treated"].eq(0), var], errors="coerce").dropna()

    mt = t.mean()
    mc = c.mean()

    vt = t.var(ddof=1)
    vc = c.var(ddof=1)

    denom = np.sqrt((vt + vc) / 2)

    nd = np.nan
    if pd.notna(denom) and denom != 0:
        nd = (mt - mc) / denom

    return {
        "csb_mean": mt,
        "control_mean": mc,
        "norm_diff": nd,
        "n_csb": len(t),
        "n_control": len(c),
    }


def make_unweighted_balance_table(df, balance_vars, buffer_kms=[10, 20, 30, 40]):
    rows = []

    for var in balance_vars:
        row = {
            "Variable": variable_label(var),
            "source_column": var,
        }

        for km in buffer_kms:
            d = df[df["buffer_km"].eq(km)].copy()
            s = unweighted_stats(d, var)

            row[f"N_CSB_{km}KM"] = s["n_csb"]
            row[f"N_Control_{km}KM"] = s["n_control"]
            row[f"ND_{km}KM"] = s["norm_diff"]

        rows.append(row)

    return pd.DataFrame(rows)


balance_unweighted = make_unweighted_balance_table(balance_source, balance_vars)

display(balance_unweighted)

,Variable,source_column,N_CSB_10KM,N_Control_10KM,ND_10KM,N_CSB_20KM,N_Control_20KM,ND_20KM,N_CSB_30KM,N_Control_30KM,ND_30KM,N_CSB_40KM,N_Control_40KM,ND_40KM
0,latitude,latitude_mean,0,5443,NaN,0,8319,NaN,0,8963,NaN,0,9092,NaN
1,longitude,longitude_mean,0,5443,NaN,0,8319,NaN,0,8963,NaN,0,9092,NaN
2,index_clean,index_clean_mean,0,5443,NaN,0,8319,NaN,0,8963,NaN,0,9092,NaN
3,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC15A__TOTAL_HH_MEMBERS_15A_mean,7,5438,-0.383359,15,8319,-0.060449,17,8963,-0.194176,19,9092,0.204404
4,GSEC12__H12Q01,GSEC12__H12Q01_mean,7,5197,0.605318,14,8281,0.258303,16,8939,-0.077373,19,9087,0.291021
5,GSEC10__H10Q09,GSEC10__H10Q09_mean,2,462,-1.744299,2,1497,-1.827651,2,2873,-1.742020,2,4114,-1.663467
6,Road type A,H18Q1A_mean,7,5441,0.829603,16,8318,0.595191,17,8963,0.839419,18,9092,0.866841
7,Road type B,H18Q1B_mean,7,5441,-1.137130,16,8318,-0.114771,17,8963,-0.105501,18,9092,-0.168139
8,Road type C,H18Q1C_mean,7,5441,0.447128,16,8318,-0.549746,17,8963,-0.531784,18,9092,-0.397380
9,Road type D,H18Q1D_mean,7,5441,-0.663678,16,8318,-0.638577,17,8963,-1.050066,18,9092,-0.881612


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


def add_att_weights_by_buffer(df, pscore_vars, buffer_kms=[10, 20, 30, 40], use_common_support=True):
    pieces = []

    for km in buffer_kms:
        d = df[df["buffer_km"].eq(km)].copy()

        usable_vars = []

        for col in pscore_vars:
            x = pd.to_numeric(d[col], errors="coerce")
            if x.notna().sum() >= 5 and x.nunique(dropna=True) >= 2:
                d[col] = x
                usable_vars.append(col)

        if not d["treated"].isin([0, 1]).all():
            d = d[d["treated"].isin([0, 1])].copy()

        if d["treated"].nunique() < 2 or len(usable_vars) == 0:
            d["pscore"] = np.nan
            d["att_weight"] = np.nan
            pieces.append(d)
            continue

        X = d[usable_vars]
        y = d["treated"].astype(int)

        model = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(max_iter=5000, solver="lbfgs")
        )

        model.fit(X, y)

        p = model.predict_proba(X)[:, 1]
        p = np.clip(p, 0.001, 0.999)

        d["pscore"] = p
        d["att_weight"] = np.where(d["treated"].eq(1), 1.0, p / (1 - p))

        if use_common_support:
            t_min = d.loc[d["treated"].eq(1), "pscore"].min()
            t_max = d.loc[d["treated"].eq(1), "pscore"].max()

            outside_support = (
                d["treated"].eq(0)
                & ~d["pscore"].between(t_min, t_max)
            )

            d.loc[outside_support, "att_weight"] = 0

        print(f"{km}KM p-score variables used:", len(usable_vars))

        pieces.append(d)

    return pd.concat(pieces, ignore_index=True, sort=False)


def weighted_mean_var_n(df, var, weight_col):
    x = pd.to_numeric(df[var], errors="coerce")
    w = pd.to_numeric(df[weight_col], errors="coerce")

    ok = x.notna() & w.notna() & w.gt(0)

    x = x[ok].astype(float)
    w = w[ok].astype(float)

    if len(x) == 0:
        return np.nan, np.nan, 0

    mean = np.average(x, weights=w)
    var = np.average((x - mean) ** 2, weights=w)

    return mean, var, len(x)


def weighted_stats(df, var, weight_col="att_weight"):
    treated = df[df["treated"].eq(1)]
    control = df[df["treated"].eq(0)]

    mt, vt, nt = weighted_mean_var_n(treated, var, weight_col)
    mc, vc, nc = weighted_mean_var_n(control, var, weight_col)

    denom = np.sqrt((vt + vc) / 2)

    nd = np.nan
    if pd.notna(denom) and denom != 0:
        nd = (mt - mc) / denom

    return {
        "csb_mean": mt,
        "control_mean": mc,
        "norm_diff": nd,
        "n_csb": nt,
        "n_control": nc,
    }


def make_weighted_balance_table(df, balance_vars, buffer_kms=[10, 20, 30, 40]):
    rows = []

    for var in balance_vars:
        row = {
            "Variable": variable_label(var),
            "source_column": var,
        }

        for km in buffer_kms:
            d = df[df["buffer_km"].eq(km)].copy()
            s = weighted_stats(d, var, weight_col="att_weight")

            row[f"N_CSB_{km}KM"] = s["n_csb"]
            row[f"N_Control_{km}KM"] = s["n_control"]
            row[f"ND_{km}KM"] = s["norm_diff"]

        rows.append(row)

    return pd.DataFrame(rows)

In [11]:
def sigmoid(z):
    z = np.clip(z, -30, 30)
    return 1 / (1 + np.exp(-z))


def fit_ridge_logit_numpy(X, y, l2=1.0, max_iter=100, tol=1e-7):
    """
couldn't get sklearn library so had to create manual logistic regression, used "https://www.kaggle.com/code/sagira/logistic-regression-math-behind-without-sklearn"
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    n, p = X.shape

    X_design = np.column_stack([np.ones(n), X])

    beta = np.zeros(p + 1)

    penalty = np.eye(p + 1)
    penalty[0, 0] = 0

    for _ in range(max_iter):
        eta = X_design @ beta
        prob = sigmoid(eta)

        w = prob * (1 - prob)
        w = np.clip(w, 1e-6, None)

        grad = X_design.T @ (y - prob) - l2 * penalty @ beta

        hess = -(X_design.T * w) @ X_design - l2 * penalty

        try:
            step = np.linalg.solve(hess, grad)
        except np.linalg.LinAlgError:
            step = np.linalg.pinv(hess) @ grad

        beta_new = beta - step

        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            break

        beta = beta_new

    return beta


def predict_ridge_logit_numpy(X, beta):
    X = np.asarray(X, dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    return sigmoid(X_design @ beta)


def prep_pscore_matrix(d, pscore_vars):
    d = d.copy()
    usable_vars = []

    for col in pscore_vars:
        if col not in d.columns:
            continue

        x = pd.to_numeric(d[col], errors="coerce")

        if x.notna().sum() >= 5 and x.nunique(dropna=True) >= 2:
            d[col] = x
            usable_vars.append(col)

    if len(usable_vars) == 0:
        return None, [], d

    X = d[usable_vars].copy()

    for col in usable_vars:
        med = X[col].median()
        X[col] = X[col].fillna(med)

    means = X.mean()
    sds = X.std(ddof=0).replace(0, 1)

    X = (X - means) / sds

    return X, usable_vars, d


def add_att_weights_by_buffer(df, pscore_vars, buffer_kms=[10, 20, 30, 40], use_common_support=True):
    pieces = []

    for km in buffer_kms:
        d = df[df["buffer_km"].eq(km)].copy()
        d = d[d["treated"].isin([0, 1])].copy()

        if d["treated"].nunique() < 2:
            d["pscore"] = np.nan
            d["att_weight"] = np.nan
            pieces.append(d)
            continue

        X, usable_vars, d = prep_pscore_matrix(d, pscore_vars)

        if X is None or len(usable_vars) == 0:
            d["pscore"] = np.nan
            d["att_weight"] = np.nan
            pieces.append(d)
            continue

        y = d["treated"].astype(int).to_numpy()

        beta = fit_ridge_logit_numpy(
            X,
            y,
            l2=2.0,
            max_iter=100,
        )

        p = predict_ridge_logit_numpy(X, beta)
        p = np.clip(p, 0.001, 0.999)

        d["pscore"] = p
        d["att_weight"] = np.where(
            d["treated"].eq(1),
            1.0,
            d["pscore"] / (1 - d["pscore"])
        )

        if use_common_support:
            t_min = d.loc[d["treated"].eq(1), "pscore"].min()
            t_max = d.loc[d["treated"].eq(1), "pscore"].max()

            outside_support = (
                d["treated"].eq(0)
                & ~d["pscore"].between(t_min, t_max)
            )

            d.loc[outside_support, "att_weight"] = 0

        print(
            f"{km}KM:",
            "pscore vars used =", len(usable_vars),
            "| treated =", int(d["treated"].eq(1).sum()),
            "| controls =", int(d["treated"].eq(0).sum()),
            "| controls with positive ATT weight =",
            int((d["treated"].eq(0) & d["att_weight"].gt(0)).sum())
        )

        pieces.append(d)

    return pd.concat(pieces, ignore_index=True, sort=False)

def weighted_mean_var_n(df, var, weight_col):
    x = pd.to_numeric(df[var], errors="coerce")
    w = pd.to_numeric(df[weight_col], errors="coerce")

    ok = x.notna() & w.notna() & w.gt(0)

    x = x[ok].astype(float)
    w = w[ok].astype(float)

    if len(x) == 0:
        return np.nan, np.nan, 0

    mean = np.average(x, weights=w)
    var = np.average((x - mean) ** 2, weights=w)

    return mean, var, len(x)


def weighted_stats(df, var, weight_col="att_weight"):
    treated = df[df["treated"].eq(1)]
    control = df[df["treated"].eq(0)]

    mt, vt, nt = weighted_mean_var_n(treated, var, weight_col)
    mc, vc, nc = weighted_mean_var_n(control, var, weight_col)

    denom = np.sqrt((vt + vc) / 2)

    nd = np.nan
    if pd.notna(denom) and denom != 0:
        nd = (mt - mc) / denom

    return {
        "csb_mean": mt,
        "control_mean": mc,
        "norm_diff": nd,
        "n_csb": nt,
        "n_control": nc,
    }


def make_weighted_balance_table(df, balance_vars, buffer_kms=[10, 20, 30, 40]):
    rows = []

    for var in balance_vars:
        row = {
            "Variable": variable_label(var),
            "source_column": var,
        }

        for km in buffer_kms:
            d = df[df["buffer_km"].eq(km)].copy()
            s = weighted_stats(d, var, weight_col="att_weight")

            row[f"N_CSB_{km}KM"] = s["n_csb"]
            row[f"N_Control_{km}KM"] = s["n_control"]
            row[f"ND_{km}KM"] = s["norm_diff"]

        rows.append(row)

    return pd.DataFrame(rows)

In [12]:
PSCORE_VARS = balance_vars.copy()

balance_source_att = add_att_weights_by_buffer(
    balance_source,
    pscore_vars=PSCORE_VARS,
    use_common_support=True,
)

balance_att = make_weighted_balance_table(
    balance_source_att,
    balance_vars,
)

display(balance_att)

10KM: pscore vars used = 38 | treated = 8 | controls = 5443 | controls with positive ATT weight = 382
20KM: pscore vars used = 38 | treated = 16 | controls = 8319 | controls with positive ATT weight = 8319
30KM: pscore vars used = 38 | treated = 17 | controls = 8963 | controls with positive ATT weight = 8963
40KM: pscore vars used = 38 | treated = 19 | controls = 9092 | controls with positive ATT weight = 9092


,Variable,source_column,N_CSB_10KM,N_Control_10KM,ND_10KM,N_CSB_20KM,N_Control_20KM,ND_20KM,N_CSB_30KM,N_Control_30KM,ND_30KM,N_CSB_40KM,N_Control_40KM,ND_40KM
0,latitude,latitude_mean,0,382,NaN,0,8319,NaN,0,8963,NaN,0,9092,NaN
1,longitude,longitude_mean,0,382,NaN,0,8319,NaN,0,8963,NaN,0,9092,NaN
2,index_clean,index_clean_mean,0,382,NaN,0,8319,NaN,0,8963,NaN,0,9092,NaN
3,GSEC15A__TOTAL_HH_MEMBERS_15A,GSEC15A__TOTAL_HH_MEMBERS_15A_mean,7,382,-0.171711,15,8319,0.139621,17,8963,-0.109733,19,9092,0.121062
4,GSEC12__H12Q01,GSEC12__H12Q01_mean,7,318,-0.375785,14,8281,0.215561,16,8939,-0.066811,19,9087,0.128542
5,GSEC10__H10Q09,GSEC10__H10Q09_mean,2,9,NaN,2,1497,-1.735713,2,2873,-1.385590,2,4114,-1.381989
6,Road type A,H18Q1A_mean,7,382,0.057996,16,8318,0.241565,17,8963,0.410936,18,9092,0.257193
7,Road type B,H18Q1B_mean,7,382,-0.728749,16,8318,-0.148525,17,8963,-0.046465,18,9092,-0.012097
8,Road type C,H18Q1C_mean,7,382,0.228508,16,8318,-0.346176,17,8963,-0.282869,18,9092,-0.246766
9,Road type D,H18Q1D_mean,7,382,0.153940,16,8318,-0.232501,17,8963,-0.442300,18,9092,-0.392419


In [13]:
out_file = OUT / "balance_tables_wave1_all_buffers.xlsx"

with pd.ExcelWriter(out_file) as writer:
    balance_unweighted.to_excel(writer, sheet_name="Unweighted_ND", index=False)
    balance_att.to_excel(writer, sheet_name="ATT_weighted_ND", index=False)
    balance_source_att.to_excel(writer, sheet_name="Panel_with_ATT_weights", index=False)

print("Saved:", out_file)

Saved: C:\Users\Carl\Desktop\RA Data\11K CANDIDATES\buffer_panels_wave1\balance_tables_wave1_all_buffers.xlsx
